# 10 - Pipeline Text-to-SQL Remoto Volve (Otimizado para Custo, Clareza e Observabilidade)

In [4]:
# Dicionário de dados operacional da base analítica usada pelo pipeline.
COLUMN_DATA_DICTIONARY = {
    "DATEPRD": {"descricao": "Data da produção/operação diária", "tipo": "datetime", "unidade": "data", "equipamento": "Historian", "natureza": "tempo", "local": "N/A"},
    "WELL_BORE_CODE": {"descricao": "Código interno do poço", "tipo": "string", "unidade": "N/A", "equipamento": "Sistema de cadastro corporativo", "natureza": "identificação", "local": "N/A"},
    "NPD_WELL_BORE_CODE": {"descricao": "Código oficial do poço na NPD", "tipo": "inteiro", "unidade": "N/A", "equipamento": "Sistema de cadastro corporativo", "natureza": "identificação", "local": "N/A"},
    "NPD_WELL_BORE_NAME": {"descricao": "Nome oficial do poço na NPD", "tipo": "string", "unidade": "N/A", "equipamento": "Sistema de cadastro corporativo", "natureza": "identificação", "local": "N/A"},
    "NPD_FIELD_CODE": {"descricao": "Código oficial do campo", "tipo": "inteiro", "unidade": "N/A", "equipamento": "Sistema de cadastro corporativo", "natureza": "identificação", "local": "N/A"},
    "NPD_FIELD_NAME": {"descricao": "Nome do campo petrolífero", "tipo": "string", "unidade": "N/A", "equipamento": "Sistema de cadastro corporativo", "natureza": "identificação", "local": "N/A"},
    "NPD_FACILITY_CODE": {"descricao": "Código da instalação offshore", "tipo": "inteiro", "unidade": "N/A", "equipamento": "Sistema de cadastro corporativo", "natureza": "identificação", "local": "N/A"},
    "NPD_FACILITY_NAME": {"descricao": "Nome da instalação/FPSO/plataforma", "tipo": "string", "unidade": "N/A", "equipamento": "Sistema de cadastro corporativo", "natureza": "identificação", "local": "N/A"},
    "ON_STREAM_HRS": {"descricao": "Horas em operação no dia", "tipo": "float", "unidade": "horas", "equipamento": "Sistema supervisório", "natureza": "tempo", "local": "Poço / sistema de produção"},
    "AVG_DOWNHOLE_PRESSURE": {"descricao": "Pressão média no fundo do poço", "tipo": "float", "unidade": "bar(a)", "equipamento": "Gauge de fundo", "natureza": "pressão", "local": "Fundo do poço"},
    "AVG_DOWNHOLE_TEMPERATURE": {"descricao": "Temperatura média no fundo do poço", "tipo": "float", "unidade": "°C", "equipamento": "Sensor downhole", "natureza": "temperatura", "local": "Fundo do poço"},
    "AVG_DP_TUBING": {"descricao": "Delta de pressão médio no tubing", "tipo": "float", "unidade": "bar", "equipamento": "Sensor de pressão do tubing", "natureza": "pressão", "local": "Tubing de produção"},
    "AVG_ANNULUS_PRESS": {"descricao": "Pressão média do anular", "tipo": "float", "unidade": "bar", "equipamento": "Sensor do anular", "natureza": "pressão", "local": "Espaço anular"},
    "AVG_CHOKE_SIZE_P": {"descricao": "Abertura média do choke", "tipo": "float", "unidade": "%", "equipamento": "Sensor do choke", "natureza": "abertura", "local": "Choke de superfície"},
    "AVG_CHOKE_UOM": {"descricao": "Unidade de medida do choke", "tipo": "string", "unidade": "texto", "equipamento": "Sensor do choke", "natureza": "unidade de abertura", "local": "Choke de superfície"},
    "AVG_WHP_P": {"descricao": "Pressão média na cabeça do poço", "tipo": "float", "unidade": "bar", "equipamento": "Sensor wellhead", "natureza": "pressão", "local": "Cabeça do poço"},
    "AVG_WHT_P": {"descricao": "Temperatura média na cabeça do poço", "tipo": "float", "unidade": "°C", "equipamento": "Sensor wellhead", "natureza": "temperatura", "local": "Cabeça do poço"},
    "DP_CHOKE_SIZE": {"descricao": "Delta de pressão associado ao choke", "tipo": "float", "unidade": "bar", "equipamento": "Sensores do choke", "natureza": "pressão", "local": "Linha do choke"},
    "BORE_OIL_VOL": {"descricao": "Volume diário de óleo produzido", "tipo": "float", "unidade": "Sm3/d", "equipamento": "Medidor multifásico", "natureza": "vazão", "local": "Linha de produção / separador"},
    "BORE_GAS_VOL": {"descricao": "Volume diário de gás produzido", "tipo": "float", "unidade": "Sm3/d", "equipamento": "Medidor de gás", "natureza": "vazão", "local": "Linha de gás / separador"},
    "BORE_WAT_VOL": {"descricao": "Volume diário de água produzida", "tipo": "float", "unidade": "Sm3/d", "equipamento": "Medidor multifásico", "natureza": "vazão", "local": "Linha de produção / separador"},
    "BORE_WI_VOL": {"descricao": "Volume diário de água injetada", "tipo": "float", "unidade": "Sm3/d", "equipamento": "Medidor de injeção", "natureza": "vazão", "local": "Linha de injeção de água"},
    "FLOW_KIND": {"descricao": "Tipo de fluxo/operação do poço", "tipo": "string", "unidade": "N/A", "equipamento": "Sistema supervisório", "natureza": "classificação operacional", "local": "N/A"},
    "WELL_TYPE": {"descricao": "Tipo do poço", "tipo": "string", "unidade": "N/A", "equipamento": "Sistema de engenharia de produção", "natureza": "classificação operacional", "local": "N/A"},
    "diff_dias": {"descricao": "Diferença de dias entre registros consecutivos", "tipo": "float", "unidade": "dias", "equipamento": "Historian", "natureza": "tempo", "local": "N/A"},
}

def _register_column(column_name, descricao, unidade, equipamento, natureza, local, tipo="float"):
    COLUMN_DATA_DICTIONARY[column_name] = {
        "descricao": descricao,
        "tipo": tipo,
        "unidade": unidade,
        "equipamento": equipamento,
        "natureza": natureza,
        "local": local,
    }

_PHASE_CONFIG = {
    "oil": {"nome": "óleo", "equipamento": "Medidor multifásico", "local": "Linha de produção / separador"},
    "gas": {"nome": "gás", "equipamento": "Medidor de gás", "local": "Linha de gás / separador"},
    "water": {"nome": "água", "equipamento": "Medidor multifásico", "local": "Linha de produção / separador"},
}

for phase, config in _PHASE_CONFIG.items():
    nome = config["nome"]
    equipamento = config["equipamento"]
    local = config["local"]

    for lag in (1, 3, 7, 14, 30):
        _register_column(f"{phase}_lag_{lag}", f"Valor defasado em {lag} períodos da produção de {nome}", "Sm3/d", equipamento, "vazão", local)

    for window in (3, 7, 14, 30):
        _register_column(f"{phase}_roll_mean_{window}", f"Média móvel da produção de {nome} em janela de {window} períodos", "Sm3/d", equipamento, "vazão", local)
        _register_column(f"{phase}_ewma_{window}", f"Média móvel exponencial da produção de {nome} em janela de {window} períodos", "Sm3/d", equipamento, "vazão suavizada", local)

    for window in (7, 14, 30):
        _register_column(f"{phase}_roll_std_{window}", f"Desvio padrão móvel da produção de {nome} em janela de {window} períodos", "Sm3/d", equipamento, "desvio padronizado", local)

    for days in (1, 3, 7):
        label = "dia" if days == 1 else "dias"
        _register_column(f"{phase}_delta_{days}d", f"Variação absoluta da produção de {nome} em {days} {label}", "Sm3/d", equipamento, "variação de vazão", local)

    for days in (1, 7, 14):
        label = "dia" if days == 1 else "dias"
        _register_column(f"{phase}_pct_change_{days}d", f"Variação percentual da produção de {nome} em {days} {label}", "%", equipamento, "variação percentual", local)

    _register_column(f"{phase}_expanding_mean", f"Média acumulada expandida da produção de {nome}", "Sm3/d", equipamento, "vazão", local)
    _register_column(f"{phase}_expanding_std", f"Desvio padrão acumulado expandido da produção de {nome}", "Sm3/d", equipamento, "desvio padronizado", local)
    _register_column(f"{phase}_cumulative", f"Volume acumulado de {nome}", "Sm3", equipamento, "volume", local)
    _register_column(f"{phase}_trend_strength", f"Intensidade da tendência da produção de {nome}", "adimensional", equipamento, "tendência", local)
    _register_column(f"{phase}_vs_trend", f"Desvio da produção de {nome} em relação à tendência", "Sm3/d", equipamento, "desvio de tendência", local)

for phase, nome, equipamento, local in (
    ("oil", "óleo", "Medidor multifásico", "Linha de produção / separador"),
    ("gas", "gás", "Medidor de gás", "Linha de gás / separador"),
):
    _register_column(f"{phase}_velocity", f"Velocidade de variação da produção de {nome}", "Sm3/dia²", equipamento, "velocidade", local)
    _register_column(f"{phase}_acceleration", f"Aceleração da variação da produção de {nome}", "Sm3/dia³", equipamento, "aceleração", local)
    _register_column(f"{phase}_volatility_index", f"Índice de volatilidade da produção de {nome}", "adimensional", equipamento, "volatilidade", local)

_register_column("oil_momentum_7d", "Momentum da produção de óleo em 7 dias", "adimensional", "Medidor multifásico", "momentum", "Linha de produção / separador")
_register_column("oil_momentum_30d", "Momentum da produção de óleo em 30 dias", "adimensional", "Medidor multifásico", "momentum", "Linha de produção / separador")
_register_column("oil_roc_7d", "Taxa de variação da produção de óleo em 7 dias", "%", "Medidor multifásico", "taxa de variação", "Linha de produção / separador")
_register_column("oil_roc_30d", "Taxa de variação da produção de óleo em 30 dias", "%", "Medidor multifásico", "taxa de variação", "Linha de produção / separador")
_register_column("oil_zscore_30", "Z-score da produção de óleo em janela de 30 períodos", "adimensional", "Medidor multifásico", "desvio padronizado", "Linha de produção / separador")


In [5]:
import os
import re
import time
import sqlite3
import numpy as np
import pandas as pd
import sqlglot
from langgraph.graph import END, START, StateGraph
from openai import OpenAI
from functools import wraps
from textwrap import dedent
from typing import Dict, Any, List
from typing_extensions import TypedDict

# =============================================================================
# VISÃO GERAL DO NOTEBOOK
# -----------------------------------------------------------------------------
# Este notebook implementa um agente Text-to-SQL para o caso Volve.
#
# A ideia central é separar o problema em 3 etapas:
# 1. Entender a pergunta em linguagem natural e gerar SQL.
# 2. Executar esse SQL com segurança no SQLite.
# 3. Transformar o resultado tabular em uma resposta operacional curta.
#
# Em outras palavras:
# pergunta humana -> SQL -> dados reais -> resposta final
#
# O notebook também mede tempo e tamanho de contexto, porque em sistemas com
# LLM isso é parte do custo e da qualidade, não apenas detalhe de engenharia.
# =============================================================================

# O AgentState é a "memória de curto prazo" do grafo.
# Cada nó do LangGraph lê e escreve algumas chaves desse dicionário.
class AgentState(TypedDict, total=False):
    question: str
    generated_sql: str
    error_message: str
    retry_count: int
    query_result: str
    query_column_context: str
    sql_generation_time: float
    sql_execution_time: float
    remote_response_time: float
    local_prompt_chars: int
    remote_prompt_chars: int
    final_response: str

MAX_SQL_RETRIES = 3
CSV_NAME = "volve_with_feature_engineering_temporal.csv"
DB_NAME = "volve_with_feature_engineering_temporal.db"
TABLE_NAME = "volve_with_feature_engineering_temporal"

# -----------------------------------------------------------------------------
# CONFIGURACAO RAPIDA DE MODELOS REMOTOS PARA TESTES
# Todo o notebook usa apenas estas duas configuracoes.
# -----------------------------------------------------------------------------
REMOTE_SQL_MODEL = "deepseek/deepseek-chat"
REMOTE_TEXT_MODEL = "google/gemini-2.5-flash"

# LOCAL_SQL_MODEL = "qwen2.5-coder:14b"
# LOCAL_TEXT_MODEL = "llama3.1:8b"

# DEBUG_METHOD_TRACE:
# - mostra quando cada método importante começa a rodar e o que ele faz.
#
# TRACE_OPERATION_METRICS:
# - mostra métricas objetivas, como tempo e tamanho de contexto.
DEBUG_METHOD_TRACE = True
TRACE_OPERATION_METRICS = True

def raw_log(message: Any) -> None:
    # Função mais "crua" possível para escrever no terminal.
    # Mantemos essa função separada para não misturar formatação com I/O.
    print(str(message), flush=True)

def safe_str(text: Any) -> str:
    # Em integrações com LLM, APIs e DataFrames, é comum receber:
    # - None
    # - bytes estranhos
    # - objetos que precisam virar texto
    #
    # Esta função centraliza a normalização de string para reduzir ruído.
    if text is None:
        return ""

    return str(text).encode("utf-8", errors="ignore").decode("utf-8")

def log_progress(message: str) -> None:
    # Todo log "normal" do pipeline passa por aqui.
    # A função safe_str evita que caracteres ruins quebrem a execução.
    raw_log(safe_str(message))

def debug_method_entry(method_name: str, purpose: str) -> None:
    # Este log didático serve para estudo.
    # Quando uma função começa, mostramos:
    # - nome da função
    # - papel dela dentro da arquitetura
    if DEBUG_METHOD_TRACE:
        raw_log(f"[DEBUG][{method_name}] {purpose}")

def trace_operation(stage: str, **metrics: Any) -> None:
    # Esta função registra métricas estruturadas.
    # Em vez de escrever texto livre, ela imprime pares chave=valor.
    #
    # Isso facilita responder perguntas como:
    # - Quanto tempo o SQL levou?
    # - O prompt ficou maior ou menor?
    # - A saída do modelo cresceu demais?
    if not TRACE_OPERATION_METRICS:
        return

    rendered_metrics = " ".join(
        f"{metric_name}={safe_str(metric_value)}"
        for metric_name, metric_value in metrics.items()
    )
    log_progress(f"[TRACE][{stage}] {rendered_metrics}".strip())

def instrument_named_method(method_name: str, purpose: str) -> None:
    # Este é um "decorador aplicado dinamicamente".
    #
    # Em vez de escrever manualmente logs de entrada e saída em dezenas de
    # funções, nós embrulhamos a função original com um wrapper que:
    # - loga a entrada
    # - mede o tempo
    # - loga o fim
    original_function = globals().get(method_name)
    if original_function is None:
        return

    @wraps(original_function)
    def wrapper(*args, **kwargs):
        # Log de início da função.
        debug_method_entry(method_name, purpose)
        started_at = time.time()
        try:
            return original_function(*args, **kwargs)
        finally:
            # Mesmo se a função der erro, o finally garante que o tempo seja
            # registrado. Isso é importante para depurar gargalos e falhas.
            elapsed = time.time() - started_at
            trace_operation(f"{method_name}.fim", elapsed_s=f"{elapsed:.4f}")

    globals()[method_name] = wrapper

def resolve_database_path(db_name: str) -> str:
    # O notebook tenta achar arquivos de dados em mais de um lugar porque,
    # dependendo de onde a célula foi executada, o diretório atual pode mudar.
    candidate_paths = [
        os.path.abspath(db_name),
        os.path.abspath(os.path.join(os.getcwd(), "notebooks", db_name)),
        os.path.abspath(os.path.join(os.getcwd(), "notebooks", "10-exercicio-production-surveillance", db_name)),
    ]

    for candidate_path in candidate_paths:
        if os.path.exists(candidate_path):
            return candidate_path

    return candidate_paths[-1]

def build_read_only_sqlite_uri(db_path: str) -> str:
    # mode=ro = read only.
    #
    # Mesmo que o LLM alucine um comando destrutivo, a conexão de leitura ajuda
    # a impedir escrita física no banco.
    return f"file:{db_path}?mode=ro"

def clean_generated_sql(raw_sql: str) -> str:
    # Muitos modelos gostam de devolver:
    # ```sql
    # SELECT ...
    # ```
    #
    # Para execução automática isso atrapalha. Aqui limpamos esse excesso.
    cleaned_sql = safe_str(raw_sql)
    for token in ("```sql", "```"):
        cleaned_sql = cleaned_sql.replace(token, "")
    return cleaned_sql.strip().rstrip(";").strip()

def format_value_for_prompt(column_name: str, value: Any) -> str:
    # Esta função prepara valores para entrar no prompt da resposta final.
    #
    # Objetivo didático importante:
    # o LLM responde melhor quando o dado chega em formato humano.
    # Exemplo:
    # - 0.15129  -> 15.13%
    # - 446.6200 -> 446.62
    if pd.isna(value):
        return "NA"

    normalized_name = safe_str(column_name).lower()

    if isinstance(value, (np.integer, int)):
        return safe_str(int(value))

    if isinstance(value, (np.floating, float)):
        numeric_value = float(value)
        if "pct" in normalized_name or "_roc_" in normalized_name:
            return f"{numeric_value * 100:.2f}%"
        return f"{numeric_value:.2f}"

    return safe_str(value)

def format_dataframe_for_prompt(df: pd.DataFrame) -> str:
    # O DataFrame é copiado para não alterar os dados originais em memória.
    # Em seguida, formatamos coluna por coluna antes de serializar para texto.
    formatted_df = df.copy()
    for column_name in formatted_df.columns:
        formatted_df[column_name] = formatted_df[column_name].map(
            lambda value, col=column_name: format_value_for_prompt(col, value)
        )
    return safe_str(formatted_df.to_string(index=False))

def extract_text_content(content: Any) -> str:
    # Alguns provedores devolvem texto simples.
    # Outros devolvem lista de blocos estruturados.
    #
    # Esta função unifica esses formatos para um texto único.
    if isinstance(content, str):
        return safe_str(content)

    if not isinstance(content, list):
        return safe_str(content)

    text_parts: List[str] = []
    for item in content:
        if isinstance(item, str):
            text_parts.append(safe_str(item))
            continue

        if not isinstance(item, dict):
            continue

        if item.get("type") != "text":
            continue

        text_value = item.get("text", "")
        if text_value:
            text_parts.append(safe_str(text_value))

    return "\n".join(part for part in text_parts if part).strip()

def extract_openrouter_text(response: Any) -> str:
    # Esta função é defensiva:
    # - detecta resposta vazia
    # - detecta HTML inesperado
    # - extrai texto de respostas no formato "choices"
    #
    # Em integrações com LLM, parsers robustos economizam muito tempo de debug.
    if isinstance(response, str):
        content = safe_str(response).strip()
        if not content:
            raise ValueError("OpenRouter retornou uma string vazia.")
        if content.lower().startswith("<!doctype html") or content.lower().startswith("<html"):
            raise ValueError(
                "OpenRouter retornou HTML em vez de texto do modelo. Verifique base_url, endpoint e credenciais."
            )
        return content

    choices = getattr(response, "choices", None)
    if not choices:
        raise ValueError("OpenRouter retornou uma resposta sem choices.")

    message = choices[0].message
    content = extract_text_content(getattr(message, "content", ""))
    if not content:
        raise ValueError("OpenRouter retornou conteúdo vazio.")
    if content.lower().startswith("<!doctype html") or content.lower().startswith("<html"):
        raise ValueError(
            "OpenRouter retornou HTML em vez de texto do modelo. Verifique base_url, endpoint e credenciais."
        )

    return content

def build_sql_generation_error(
    state: AgentState,
    error_message: str,
    elapsed: float,
    local_prompt_chars: int,
) -> Dict[str, Any]:
    # Em arquiteturas com grafo, erro também é estado.
    #
    # Em vez de simplesmente dar raise, devolvemos um dicionário que o próximo
    # nó consegue interpretar para decidir entre retry ou encerramento.
    return {
        "generated_sql": "",
        "error_message": safe_str(error_message),
        "retry_count": state.get("retry_count", 0) + 1,
        "sql_generation_time": state.get("sql_generation_time", 0.0) + elapsed,
        "local_prompt_chars": state.get("local_prompt_chars", 0) + local_prompt_chars,
    }

def build_empty_execution_response(error_message: str) -> Dict[str, Any]:
    # Esta resposta "vazia" é usada quando a etapa SQL não deve prosseguir.
    # Exemplo: guardrail bloqueou a query.
    return {
        "error_message": safe_str(error_message),
        "query_result": "",
        "query_column_context": "",
    }

def try_build_rule_based_sql(question: str) -> str:
    # Este é um atalho importante:
    # para perguntas muito simples, não vale a pena pagar custo de LLM.
    #
    # Se detectarmos uma pergunta simples de média, máximo ou mínimo usando uma
    # única coluna explícita, respondemos com SQL determinístico.
    normalized_question = safe_str(question).lower()
    explicit_columns = [
        column_name
        for column_name in SCHEMA_COLUMN_NAMES
        if column_name.lower() in normalized_question and column_name != "DATEPRD"
    ]

    if len(explicit_columns) != 1:
        return ""

    column_name = explicit_columns[0]
    normalized_words = set(
        word for word in re.sub(r"[^\\w]+", " ", normalized_question).split() if word
    )

    avg_tokens = {"media", "média"}
    max_tokens = {"maior", "máxima", "maxima", "máximo", "maximo", "pico"}
    min_tokens = {"menor", "mínima", "minima", "mínimo", "minimo"}

    # Média: devolve um único valor agregado.
    if normalized_words & avg_tokens:
        alias_name = f"avg_{column_name.lower()}"
        return f"SELECT AVG({column_name}) AS {alias_name} FROM {TABLE_NAME}"

    # Máximo: devolve a data em que o valor foi máximo e o valor.
    if normalized_words & max_tokens:
        return (
            f"SELECT DATEPRD, {column_name} "
            f"FROM {TABLE_NAME} "
            f"ORDER BY {column_name} DESC LIMIT 1"
        )

    # Mínimo: mesma lógica do máximo, invertendo a ordenação.
    if normalized_words & min_tokens:
        return (
            f"SELECT DATEPRD, {column_name} "
            f"FROM {TABLE_NAME} "
            f"ORDER BY {column_name} ASC LIMIT 1"
        )

    return ""

CSV_PATH = resolve_database_path(CSV_NAME)
DB_PATH = resolve_database_path(DB_NAME)
# Se o CSV não existir, falhamos cedo.
# Falha cedo é melhor do que deixar o erro aparecer muito depois no pipeline.
if not os.path.exists(CSV_PATH):
    raise FileNotFoundError(f"Arquivo CSV não encontrado: {CSV_PATH}")

# Em notebooks, reexecuções da célula podem deixar conexões antigas abertas.
# Por isso tentamos fechar a conexão anterior antes de recriar tudo.
if "conn" in globals():
    try:
        conn.close()
    except Exception:
        pass

# Primeiro carregamos o CSV enriquecido e materializamos um SQLite local com
# o mesmo nome-base para servir ao agente Text-to-SQL.
source_df = pd.read_csv(CSV_PATH)
if "DATEPRD" in source_df.columns:
    source_df["DATEPRD"] = pd.to_datetime(source_df["DATEPRD"], errors="coerce")

build_conn = sqlite3.connect(DB_PATH)
source_df.to_sql(TABLE_NAME, build_conn, if_exists="replace", index=False)
build_conn.close()

# setup_conn é usada apenas para ler schema e dados iniciais.
setup_conn = sqlite3.connect(DB_PATH)
source_df = pd.read_sql(f'SELECT * FROM "{TABLE_NAME}"', setup_conn)
schema_df = pd.read_sql(f"PRAGMA table_info({TABLE_NAME})", setup_conn)
# schema_text vira uma representação curta do schema para o prompt SQL.
schema_text = ", ".join(
    f"{safe_str(row['name'])} ({safe_str(row['type'] or 'TEXT')})"
    for _, row in schema_df.iterrows()
)
setup_conn.close()

# A conexão principal do pipeline é aberta em modo somente leitura.
conn = sqlite3.connect(
    build_read_only_sqlite_uri(DB_PATH),
    uri=True,
    # check_same_thread=False evita restrições desnecessárias em cenários de
    # reuso dentro do notebook e da orquestração.
    check_same_thread=False,
)

# SCHEMA_COLUMN_NAMES ajuda em validações, fast path e contexto.
SCHEMA_COLUMN_NAMES = [safe_str(column_name) for column_name in schema_df["name"].tolist()]

# SCHEMA_TYPES_BY_NAME ajuda a explicar para o LLM "o que é" cada coluna.
SCHEMA_TYPES_BY_NAME = {
    safe_str(row["name"]): safe_str(row["type"] or "TEXT")
    for _, row in schema_df.iterrows()
}

# Reusa o dicionário global definido na célula anterior.

def infer_metadata_from_column_name(column_name: str) -> Dict[str, str]:
    normalized_name = safe_str(column_name)
    if normalized_name in COLUMN_DATA_DICTIONARY:
        return COLUMN_DATA_DICTIONARY[normalized_name]

    base_info = {
        "descricao": f"Coluna analítica derivada: {normalized_name}",
        "tipo": "float",
        "unidade": "N/A",
        "equipamento": "Dataset analítico",
        "natureza": "analítica",
        "local": "N/A",
    }

    if normalized_name.startswith("oil_"):
        base_info["equipamento"] = "Medidor multifásico"
        base_info["local"] = "Linha de produção / separador"
        base_info["unidade"] = "Sm3/d"
        base_info["descricao"] = f"Indicador derivado de óleo: {normalized_name}"
    elif normalized_name.startswith("gas_"):
        base_info["equipamento"] = "Medidor de gás"
        base_info["local"] = "Linha de gás / separador"
        base_info["unidade"] = "Sm3/d"
        base_info["descricao"] = f"Indicador derivado de gás: {normalized_name}"
    elif normalized_name.startswith("water_"):
        base_info["equipamento"] = "Medidor multifásico"
        base_info["local"] = "Linha de produção / separador"
        base_info["unidade"] = "Sm3/d"
        base_info["descricao"] = f"Indicador derivado de água: {normalized_name}"

    if normalized_name.endswith("_cumulative"):
        base_info["natureza"] = "volume"
        base_info["unidade"] = "Sm3"
    elif normalized_name.endswith("_velocity"):
        base_info["natureza"] = "velocidade"
        base_info["unidade"] = "Sm3/dia"
    elif normalized_name.endswith("_acceleration"):
        base_info["natureza"] = "aceleração"
        base_info["unidade"] = "Sm3/dia²"
    elif "pct_change" in normalized_name or "_roc_" in normalized_name:
        base_info["natureza"] = "variação percentual"
        base_info["unidade"] = "%"
    elif "roll_std" in normalized_name or "expanding_std" in normalized_name:
        base_info["natureza"] = "dispersão"
    elif "volatility" in normalized_name:
        base_info["natureza"] = "volatilidade"
        base_info["unidade"] = "adimensional"
    elif "trend_strength" in normalized_name or normalized_name.endswith("_vs_trend"):
        base_info["natureza"] = "tendência"
        base_info["unidade"] = "adimensional"
    elif "momentum" in normalized_name:
        base_info["natureza"] = "momentum"
    elif "zscore" in normalized_name:
        base_info["natureza"] = "desvio padronizado"
        base_info["unidade"] = "adimensional"
    elif "lag" in normalized_name or "roll_mean" in normalized_name or "ewma" in normalized_name or "delta" in normalized_name or "expanding_mean" in normalized_name:
        base_info["natureza"] = "vazão"

    return base_info

def build_dictionary_context_for_columns(column_names: List[str], title: str) -> str:
    lines = [title]
    for column_name in column_names:
        normalized_name = safe_str(column_name)
        metadata = infer_metadata_from_column_name(normalized_name)
        sql_type = SCHEMA_TYPES_BY_NAME.get(normalized_name, metadata.get("tipo", "TEXT"))
        lines.append(
            f"- {normalized_name} | sql_type={sql_type} | descricao={metadata['descricao']} | unidade={metadata['unidade']} | natureza={metadata['natureza']} | equipamento={metadata['equipamento']} | local={metadata['local']}"
        )
    return "\n".join(lines)

# Este dicionário é uma ponte entre linguagem humana e schema técnico.
#
# Ideia didática importante:
# o operador fala "pressão de fundo";
# o banco conhece "AVG_DOWNHOLE_PRESSURE".
#
# O LLM trabalha melhor quando ajudamos a fazer esse mapeamento sem expor o
# schema cru ao usuário final.
QUESTION_KEYWORD_HINTS = {
    "oleo": ["DATEPRD", "BORE_OIL_VOL", "oil_roll_30", "oil_pct_change_1d", "oil_momentum_30d"],
    "óleo": ["DATEPRD", "BORE_OIL_VOL", "oil_roll_30", "oil_pct_change_1d", "oil_momentum_30d"],
    "agua": ["DATEPRD", "BORE_WAT_VOL", "water_trend_strength", "water_cumulative", "water_vs_trend"],
    "água": ["DATEPRD", "BORE_WAT_VOL", "water_trend_strength", "water_cumulative", "water_vs_trend"],
    "produção": ["DATEPRD", "BORE_OIL_VOL", "BORE_WAT_VOL", "ON_STREAM_HRS"],
    "producao": ["DATEPRD", "BORE_OIL_VOL", "BORE_WAT_VOL", "ON_STREAM_HRS"],
    "gas": ["DATEPRD", "gas_lag_30"],
    "press": ["DATEPRD", "AVG_DOWNHOLE_PRESSURE", "AVG_DP_TUBING", "AVG_WHP_P"],
    "pressão": ["DATEPRD", "AVG_DOWNHOLE_PRESSURE", "AVG_DP_TUBING", "AVG_WHP_P"],
    "pressao": ["DATEPRD", "AVG_DOWNHOLE_PRESSURE", "AVG_DP_TUBING", "AVG_WHP_P"],
    "fundo": ["DATEPRD", "AVG_DOWNHOLE_PRESSURE", "AVG_DP_TUBING", "AVG_WHP_P"],
    "tubing": ["DATEPRD", "AVG_DOWNHOLE_PRESSURE", "AVG_DP_TUBING"],
    "temper": ["DATEPRD", "AVG_DOWNHOLE_TEMPERATURE", "AVG_WHT_P"],
    "choke": ["DATEPRD", "AVG_CHOKE_SIZE_P"],
    "horas": ["DATEPRD", "ON_STREAM_HRS"],
    "operação": ["DATEPRD", "ON_STREAM_HRS"],
    "operacao": ["DATEPRD", "ON_STREAM_HRS"],
    "data": ["DATEPRD"],
    "dia": ["DATEPRD"],
    "média": ["DATEPRD", "BORE_OIL_VOL", "oil_roll_30", "oil_pct_change_1d", "oil_zscore_30"],
    "media": ["DATEPRD", "BORE_OIL_VOL", "oil_roll_30", "oil_pct_change_1d", "oil_zscore_30"],
    "padrão": ["DATEPRD", "BORE_OIL_VOL", "oil_roll_30", "oil_zscore_30"],
    "padrao": ["DATEPRD", "BORE_OIL_VOL", "oil_roll_30", "oil_zscore_30"],
    "anomalia": ["DATEPRD", "BORE_OIL_VOL", "oil_zscore_30"],
    "anormal": ["DATEPRD", "BORE_OIL_VOL", "oil_zscore_30"],
    "tend": ["DATEPRD", "oil_momentum_30d", "oil_roc_30d", "water_trend_strength"],
    "ritmo": ["DATEPRD", "BORE_OIL_VOL", "oil_momentum_30d", "oil_acceleration", "oil_roc_30d"],
    "aceleração": ["DATEPRD", "BORE_OIL_VOL", "oil_momentum_30d", "oil_acceleration", "oil_roc_30d"],
    "aceleracao": ["DATEPRD", "BORE_OIL_VOL", "oil_momentum_30d", "oil_acceleration", "oil_roc_30d"],
    "previs": ["DATEPRD", "oil_momentum_30d", "oil_acceleration", "oil_roc_30d", "water_trend_strength"],
    "projeção": ["DATEPRD", "oil_momentum_30d", "oil_acceleration", "oil_roc_30d"],
    "projecao": ["DATEPRD", "oil_momentum_30d", "oil_acceleration", "oil_roc_30d"],
    "estimativa": ["DATEPRD", "oil_momentum_30d", "oil_acceleration", "oil_roc_30d"],
    "risco": ["DATEPRD", "water_trend_strength", "water_cumulative", "water_vs_trend", "oil_acceleration"],
    "declínio": ["DATEPRD", "BORE_OIL_VOL", "oil_expanding_mean", "oil_expanding_std", "oil_volatility_index", "oil_roc_30d"],
    "declinio": ["DATEPRD", "BORE_OIL_VOL", "oil_expanding_mean", "oil_expanding_std", "oil_volatility_index", "oil_roc_30d"],
    "oscilação": ["DATEPRD", "BORE_OIL_VOL", "oil_expanding_std", "oil_volatility_index"],
    "oscilacao": ["DATEPRD", "BORE_OIL_VOL", "oil_expanding_std", "oil_volatility_index"],
    "estabilização": ["DATEPRD", "BORE_OIL_VOL", "oil_expanding_mean", "oil_volatility_index", "oil_roc_30d"],
    "estabilizacao": ["DATEPRD", "BORE_OIL_VOL", "oil_expanding_mean", "oil_volatility_index", "oil_roc_30d"],
    "instabilidade": ["DATEPRD", "AVG_DOWNHOLE_PRESSURE", "AVG_DP_TUBING", "oil_volatility_index", "oil_acceleration"],
    "mecânica": ["DATEPRD", "AVG_DOWNHOLE_PRESSURE", "AVG_DP_TUBING", "oil_volatility_index", "oil_acceleration"],
    "mecanica": ["DATEPRD", "AVG_DOWNHOLE_PRESSURE", "AVG_DP_TUBING", "oil_volatility_index", "oil_acceleration"],
}

def build_local_sql_context(question: str) -> str:
    # Esta função reduz o contexto SQL.
    #
    # Em vez de mandar o schema inteiro "sem pensar", tentamos destacar só as
    # colunas mais prováveis para a pergunta atual.
    normalized_question = safe_str(question).lower()
    relevant_columns: List[str] = []

    # Procuramos palavras-chave humanas na pergunta.
    for keyword, columns in QUESTION_KEYWORD_HINTS.items():
        if keyword in normalized_question:
            for column_name in columns:
                if column_name in SCHEMA_TYPES_BY_NAME and column_name not in relevant_columns:
                    relevant_columns.append(column_name)

    # Se nada bater, fazemos um fallback pequeno em vez de mandar tudo.
    if not relevant_columns:
        relevant_columns = SCHEMA_COLUMN_NAMES[:12]

    lines = [
        "CONTEXTO ENXUTO PARA GERAR SQL:",
        "- O banco operacional carregado neste notebook e volve_ml_ready.db.",
        f"- A tabela principal e {TABLE_NAME}.",
        "- Use somente nomes de colunas existentes no schema abaixo.",
        "",
        "COLUNAS MAIS RELEVANTES PARA ESTA PERGUNTA:",
    ]

    for column_name in relevant_columns:
        lines.append(f"- {column_name} ({SCHEMA_TYPES_BY_NAME.get(column_name, 'TEXT')})")

    lines.append("")
    lines.append(build_dictionary_context_for_columns(relevant_columns, "DICIONÁRIO OPERACIONAL DAS COLUNAS RELEVANTES:"))

    return "\n".join(lines)

def build_query_column_context(sql: str, columns: List[str]) -> str:
    # Depois que o SQL roda, esta função explica ao modelo de resposta o que
    # cada coluna retornada representa e se ela veio do schema ou é um alias.
    lines = ["COLUNAS RETORNADAS PELA CONSULTA:"]
    for column_name in columns:
        normalized_name = safe_str(column_name)
        sql_type = SCHEMA_TYPES_BY_NAME.get(normalized_name, "RESULT")
        origem = "schema" if normalized_name in SCHEMA_TYPES_BY_NAME else "alias_ou_expressao"
        metadata = infer_metadata_from_column_name(normalized_name)
        lines.append(
            f"- {normalized_name} | sql_type={sql_type} | origem={origem} | descricao={metadata['descricao']} | unidade={metadata['unidade']} | natureza={metadata['natureza']} | equipamento={metadata['equipamento']} | local={metadata['local']}"
        )

    return "\n".join(lines)

log_progress(f"[DADOS] CSV operacional carregado de: {CSV_PATH}")
log_progress(f"[DADOS] Banco SQLite materializado em: {DB_PATH}")
log_progress(f"[DADOS] Tabela operacional: {TABLE_NAME}")
log_progress(f"[DADOS] Registros carregados em source_df: {len(source_df)}")

# -----------------------------------------------------------------------------
# 1. CONFIGURAÇÃO DE SEGURANÇA E CONEXÃO OPENROUTER
# -----------------------------------------------------------------------------
OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY", "SUA_CHAVE_OPENROUTER_AQUI")

# PIPELINE_AUTO_RUN controla se o notebook executa o fluxo automaticamente ao
# rodar a célula. Em estudo, isso é útil para alternar entre:
# - modo automático
# - modo exploratório/manual
PIPELINE_AUTO_RUN = os.getenv("NOTEBOOK_09_AUTO_RUN", "1").strip() != "0"

# Banco interno de perguntas de teste.
# Repare que aqui a linguagem é humana, não linguagem de banco.
PIPELINE_QUESTION_BANK = [
    {
        "categoria": "simples",
        "pergunta": "Como estavam as produções de óleo e de água em 10/06/2014?",
    },
    {
        "categoria": "simples",
        "pergunta": "Qual foi a leitura mais recente da pressão de fundo do poço?",
    },
    {
        "categoria": "simples",
        "pergunta": "Em média, quantas horas por dia o poço ficou em operação?",
    },
    {
        "categoria": "normal",
        "pergunta": "Em que dia o poço atingiu sua maior produção de óleo e qual foi esse volume?",
    },
    {
        "categoria": "normal",
        "pergunta": "Quais foram os 5 dias com maior produção de água?",
    },
    {
        "categoria": "normal",
        "pergunta": "Nos 10 registros mais recentes, como evoluíram a data, a produção de óleo e a produção de água?",
    },
    {
        "categoria": "complexa",
        "pergunta": "A produção de óleo mais recente ficou acima ou abaixo do comportamento típico do último mês? Mostre também se houve alta ou queda no dia e se o valor parece fora do padrão.",
    },
    {
        "categoria": "complexa",
        "pergunta": "Em que datas a produção de óleo ficou claramente fora do padrão normal do último mês?",
    },
    {
        "categoria": "complexa",
        "pergunta": "O dado mais recente sugere aumento de risco de avanço de água no poço? Mostre os sinais que sustentam essa conclusão.",
    },
    {
        "categoria": "complexa",
        "pergunta": "Pelo ritmo recente de produção e pela aceleração observada, qual seria a estimativa de produção de óleo para o próximo dia e quais sinais apoiam essa leitura?",
    },
    {
        "categoria": "complexa",
        "pergunta": "Observando o histórico acumulado e a oscilação da produção, o poço está perdendo força de forma contínua ou entrando em estabilização?",
    },
    {
        "categoria": "complexa",
        "pergunta": "Com base nas pressões mais recentes e no comportamento da produção, existe sinal de instabilidade mecânica de curtíssimo prazo no poço?",
    },
]

def choose_pipeline_question() -> Dict[str, Any]:
    # Sorteia uma pergunta e também devolve o índice, o que ajuda em logs,
    # reprodutibilidade e depuração de casos específicos.
    selected_index = int(np.random.choice(len(PIPELINE_QUESTION_BANK)))
    selected_entry = dict(PIPELINE_QUESTION_BANK[selected_index])
    selected_entry["indice"] = selected_index
    return selected_entry

# Cliente unificado usando a especificação OpenAI para conectar no OpenRouter
openrouter_client = OpenAI(
    # OpenRouter expõe endpoint compatível com a API da OpenAI.
    base_url="https://openrouter.ai/api/v1",
    api_key=OPENROUTER_API_KEY,
)

# Palavras-chave proibidas estritas para o Guardrail de Escrita
# Mesmo que o prompt peça "somente SELECT", criamos uma defesa adicional.
PROHIBITED_KEYWORDS = {"DROP", "DELETE", "INSERT", "UPDATE", "ALTER", "CREATE", "TRUNCATE", "EXECUTE", "REPLACE"}

# -----------------------------------------------------------------------------
# 2. CAMADA RIGOROSA DE GUARDRAIL E VALIDAÇÃO ESTÁTICA
# -----------------------------------------------------------------------------
def detect_implausible_forecast_sql(sql_query: str) -> str:
    # Este guardrail foi criado porque o LLM pode tentar extrapolar demais.
    # Como este fluxo é SQL + contexto analítico, aceitamos apenas horizonte D+1
    # para projeções diretas.
    normalized_sql = " ".join(safe_str(sql_query).lower().split())

    # Bloqueia aliases como proj_oleo_7_dias, proj_oleo_30_dias etc.
    forecast_alias_matches = re.findall(r"\bproj_[a-z0-9_]*?(\d+)_dias\b", normalized_sql)
    for horizon_text in forecast_alias_matches:
        if int(horizon_text) > 1:
            return (
                "Guardrail preditivo: projecoes SQL para horizontes acima de D+1 foram bloqueadas. "
                "Retorne apenas D+1 e os indicadores tecnicos de suporte."
            )

    # Bloqueia uma forma específica de extrapolação quadrática que tende a
    # parecer "inteligente", mas pode ser operacionalmente irresponsável.
    if re.search(r"0\.5\s*\*\s*oil_acceleration\s*\*\s*\(\s*([2-9]\d*)\s*\*\s*\1\s*\)", normalized_sql):
        return (
            "Guardrail preditivo: extrapolacao quadratica com aceleracao para horizontes maiores que D+1 foi bloqueada. "
            "Use apenas projecao de curtissimo prazo ou retorne indicadores para analise narrativa."
        )

    return ""

def validar_sql_seguro_e_compativel(sql_query: str) -> tuple[bool, str]:
    """Retorna (True, "") se o SQL for seguro e compatível com SQLite."""

    # Primeiro validamos segurança lexical.
    # Não basta "confiar" no modelo.
    sql_limpo = sql_query.strip().upper()
    
    if any(keyword in sql_limpo for keyword in PROHIBITED_KEYWORDS):
        return False, "Bloqueio de Segurança: Comando de modificação/escrita detectado!"

    # Depois validamos plausibilidade de forecast para este caso de uso.
    forecast_guardrail_message = detect_implausible_forecast_sql(sql_query)
    if forecast_guardrail_message:
        return False, forecast_guardrail_message
        
    try:
        # sqlglot funciona aqui como um parser estático.
        # Ele ajuda a pegar erro de sintaxe antes de bater no banco.
        sqlglot.parse_one(sql_query, read="sqlite")
        return True, ""
    except sqlglot.errors.ParseError as e:
        return False, f"Erro de sintaxe estática (Dialeto SQLite): {str(e)}"

# -----------------------------------------------------------------------------
# 3. PROMPT DE ENGENHARIA DE PROMPT COM FEW-SHOT (MECÂNICA E PREVISÕES)
# -----------------------------------------------------------------------------
def build_sql_prompt(question: str, error_message: str) -> str:
    # Quando a primeira tentativa falha, o erro do SQLite volta para o modelo.
    # Isso cria um loop de autocorreção orientado por evidência real.
    retry_context = ""
    if error_message:
        retry_context = f"\nATENÇÃO: sua tentativa anterior falhou com o erro: {error_message}. Corrija a sintaxe para o SQLite."

    # Few-shot = exemplos curtos que mostram ao modelo o estilo de resposta
    # esperado. Aqui ensinamos:
    # - tipo de pergunta
    # - tipo de SQL
    # - disciplina do schema
    few_shot_examples = f"""
=== EXEMPLOS DE TRADUÇÃO DE PERGUNTAS PARA SQLITE ===

[PERGUNTA SIMPLES]
Pergunta: Qual foi a produção de óleo e água no dia 10 de junho de 2014?
SQL: SELECT DATEPRD, BORE_OIL_VOL, BORE_WAT_VOL FROM {TABLE_NAME} WHERE DATEPRD = '2014-06-10'

[PERGUNTA SÉRIE TEMPORAL - HISTÓRICO COMPREENSIVO]
Pergunta: A produção de óleo de ontem caiu muito em relação à média móvel de 30 dias? Mostre também a variação percentual de ontem.
SQL: SELECT DATEPRD, BORE_OIL_VOL, oil_lag_1, oil_roll_30, oil_pct_change_1d FROM {TABLE_NAME} ORDER BY DATEPRD DESC LIMIT 1

[PERGUNTA SÉRIE TEMPORAL - DETECÇÃO DE ANOMALIA FISICA]
Pergunta: Em quais datas tivemos anomalias graves de produção de óleo fora do padrão estatístico dos últimos 30 dias?
SQL: SELECT DATEPRD, BORE_OIL_VOL, oil_zscore_30 FROM {TABLE_NAME} WHERE abs(oil_zscore_30) > 2.0 ORDER BY DATEPRD DESC

[PREVISÃO E PROJEÇÃO ANALÍTICA 1 - PRÓXIMOS DIAS BASEADO EM MOMENTUM]
Pergunta: Com base no momentum atual de 30 dias e na aceleração do óleo, qual é a projeção linear estimada de volume de óleo para os próximos dias se o poço mantiver o ritmo atual?
SQL: SELECT DATEPRD, BORE_OIL_VOL, oil_momentum_30d, oil_acceleration, (BORE_OIL_VOL + (oil_momentum_30d / 30.0)) AS proj_oleo_proximo_dia FROM {TABLE_NAME} ORDER BY DATEPRD DESC LIMIT 1

[PREVISÃO E PROJEÇÃO ANALÍTICA 2 - EXAUSTÃO POR EXCESSO DE ÁGUA (WATER CUT/CONING)]
Pergunta: Olhando para a força da tendência da água e o volume acumulado, o poço corre risco iminente de colapso de óleo por invasão de água se a tendência atual continuar forte?
SQL: SELECT DATEPRD, BORE_OIL_VOL, BORE_WAT_VOL, water_cumulative, water_trend_strength, water_vs_trend FROM {TABLE_NAME} WHERE water_trend_strength > 0 AND water_vs_trend > 1.0 ORDER BY DATEPRD DESC LIMIT 5

[PREVISÃO E PROJEÇÃO ANALÍTICA 3 - PREVISÃO DE DECAIMENTO DE PRODUTIVIDADE DO RESERVATÓRIO]
Pergunta: Analisando o declínio histórico acumulado (expanding mean) e a volatilidade do poço, há sinais de enfraquecimento contínuo de energia ou a produção está se estabilizando?
SQL: SELECT DATEPRD, BORE_OIL_VOL, oil_expanding_mean, oil_expanding_std, oil_volatility_index, oil_roc_30d FROM {TABLE_NAME} ORDER BY DATEPRD DESC LIMIT 10

[PREVISÃO E PROJEÇÃO ANALÍTICA 4 - RISCO DE PRESSÃO (INTEGRIDADE POÇO)]
Pergunta: Cruzando a volatilidade do poço e o delta de pressão do tubing, existe o risco do poço fechar (interrupção de fluxo) por perda severa de energia de fundo nas próximas operações?
SQL: SELECT DATEPRD, AVG_DOWNHOLE_PRESSURE, AVG_DP_TUBING, oil_volatility_index, oil_acceleration FROM {TABLE_NAME} WHERE AVG_DOWNHOLE_PRESSURE < 212.0 OR oil_acceleration < -10.0 ORDER BY DATEPRD DESC LIMIT 5
=====================================================
"""

    # Ordem do prompt:
    # 1. schema
    # 2. exemplos
    # 3. regras
    # 4. contexto local
    # 5. pergunta
    #
    # A parte mais estática fica mais acima para favorecer cache estrutural.
    prompt = f"""
Esquema:
{schema_text}

{few_shot_examples}

Sua tarefa é gerar SQL SQLite para uma série temporal real de produção offshore do projeto Volve, no Mar do Norte da Noruega.
Tabela disponível: {TABLE_NAME}

Regras Cruciais:
1. Retorne somente SQL puro, sem markdown ou blocos de código.
2. Use apenas comandos SELECT de leitura.
3. Use exatamente os nomes das colunas disponíveis no esquema.
4. Para análises de previsão ou tendências futuras, utilize as colunas matemáticas derivadas disponíveis no ecossistema (_trend_strength, _acceleration, _momentum_30d, _roc_30d, _vs_trend) para sustentar o relatório técnico.
5. Quando a pergunta pedir projeção sem horizonte explicitamente definido, retorne somente projeção de curtíssimo prazo D+1.
6. Não gere colunas de projeção multi-dia como proj_oleo_3_dias, proj_oleo_7_dias ou equivalentes.
7. Não use extrapolação quadrática com aceleração para horizontes acima de D+1.
8. Para horizontes maiores que D+1, retorne os indicadores de suporte (momentum, aceleração, roc, trend_strength, zscore) e deixe a interpretação para o parecer técnico final.

Contexto semantico enxuto:
{build_local_sql_context(question)}
{retry_context}

Pergunta: {question}
SQL:
"""
    final_prompt = dedent(prompt).strip()

    # Medimos o prompt porque, em LLM, caracteres/tokens são custo e também
    # influenciam latência.
    trace_operation(
        "build_sql_prompt.contexto",
        question_chars=len(question),
        error_chars=len(error_message),
        prompt_chars=len(final_prompt),
    )
    return final_prompt

# -----------------------------------------------------------------------------
# 4. NÓ DE GERAÇÃO SQL
# -----------------------------------------------------------------------------
# Este nó decide entre:
# - usar um atalho heurístico (fast path)
# - ou chamar o modelo remoto para gerar SQL
# -----------------------------------------------------------------------------
def generate_sql_node(state: AgentState) -> Dict[str, Any]:
    # O estado entra com a pergunta e com eventual erro anterior.
    question = safe_str(state["question"])
    retry_count = state.get("retry_count", 0)
    attempt_number = retry_count + 1

    # Primeiro tentamos o caminho barato e determinístico.
    # Se funcionar, evitamos custo de LLM.
    fast_path_sql = try_build_rule_based_sql(question)
    if fast_path_sql:
        log_progress(f"[SQL][FASTPATH] pergunta={question} sql={safe_str(fast_path_sql)}")
        return {
            "generated_sql": safe_str(fast_path_sql),
            "retry_count": retry_count + 1,
            "error_message": "",
            "sql_generation_time": state.get("sql_generation_time", 0.0),
            "local_prompt_chars": state.get("local_prompt_chars", 0),
        }

    # Se não houver fast path, seguimos para geração remota.
    error_message = safe_str(state.get("error_message", ""))
    prompt_corpo = build_sql_prompt(question, error_message)
    
    # O system prompt define "quem" o modelo deve ser.
    # Aqui o modelo deve agir como especialista em SQLite para Oil & Gas,
    # devolvendo apenas SQL executável.
    system_prompt = (
        "Você é um engenheiro sênior de dados especialista em SQLite para Oil & Gas.\n"
        "Sua saída deve conter EXCLUSIVAMENTE o código SQL limpo pronto para execução direta.\n"
        "Proibido adicionar marcações de blocos de código markdown ou explicações.\n"
        "Utilize inteligentemente as colunas analíticas e preditivas injetadas no banco."
    )

    log_progress(f"[SQL][OPENROUTER] Executando geração remota com {REMOTE_SQL_MODEL} - Tentativa {attempt_number}")
    sql_generation_start_time = time.time()
    
    try:
        # temperature=0.0 reduz criatividade e aumenta consistência estrutural,
        # algo desejável quando a saída é código SQL.
        response = openrouter_client.chat.completions.create(
            model=REMOTE_SQL_MODEL,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": prompt_corpo}
            ],
            temperature=0.0, # Zero criatividade, foco total em precisão computacional
            extra_headers={
                "HTTP-Referer": "https://petroleo.local", 
                "X-Title": "FewShot Text-to-SQL Volve",
            }
        )
        
        raw_sql = extract_openrouter_text(response)
        clean_sql = clean_generated_sql(raw_sql)
        
        elapsed = time.time() - sql_generation_start_time
        prompt_chars_estimado = len(system_prompt) + len(prompt_corpo)
        trace_operation(
            "generate_sql_node.contexto_entrada",
            model=REMOTE_SQL_MODEL,
            question_chars=len(question),
            error_chars=len(error_message),
            system_prompt_chars=len(system_prompt),
            user_prompt_chars=len(prompt_corpo),
            total_input_chars=prompt_chars_estimado,
        )
        
        log_progress(f"[SQL][OK] Query gerada por {REMOTE_SQL_MODEL} em {elapsed:.2f}s")
        trace_operation(
            "generate_sql_node.contexto_saida",
            model=REMOTE_SQL_MODEL,
            output_chars=len(clean_sql),
            elapsed_s=f"{elapsed:.2f}",
        )
        
        return {
            "generated_sql": clean_sql,
            "retry_count": attempt_number,
            "error_message": "", 
            "sql_generation_time": state.get("sql_generation_time", 0.0) + elapsed,
            "local_prompt_chars": state.get("local_prompt_chars", 0) + prompt_chars_estimado,
        }
        
    except Exception as exc:
        # Em vez de explodir o notebook, transformamos a falha em estado.
        elapsed = time.time() - sql_generation_start_time
        detail = safe_str(str(exc))
        log_progress(f"[SQL][ERRO] Falha na API OpenRouter: {detail}")
        return build_sql_generation_error(state, f"Erro na API OpenRouter: {detail}", elapsed, 0)

# -----------------------------------------------------------------------------
# 5. NÓ DE EXECUÇÃO SQL
# -----------------------------------------------------------------------------
# Aqui o SQL já foi gerado. Este nó:
# - bloqueia query insegura
# - executa no SQLite
# - prepara o resultado para o modelo final
# -----------------------------------------------------------------------------
def execute_sql_node(state: AgentState) -> Dict[str, Any]:
    error_message = safe_str(state.get("error_message", ""))

    # Se o erro veio da API remota, não faz sentido tentar rodar SQL vazio.
    if "Erro na API OpenRouter" in error_message:
        return {"query_result": ""}
        
    generated_sql = safe_str(state.get("generated_sql", "")).strip()
    if not generated_sql:
        return build_empty_execution_response("Nenhum SQL foi gerado pelo modelo.")
        
    # Primeiro rodamos os guardrails locais.
    eh_valido_e_seguro, motivo_erro = validar_sql_seguro_e_compativel(generated_sql)
    if not eh_valido_e_seguro:
        log_progress(f"(GUARDRAIL)(BLOQUEIO) {motivo_erro} na query: {generated_sql}")
        return build_empty_execution_response(motivo_erro)

    log_progress(f"(SQLITE) Executando query no banco operacional.")
    sql_execution_start_time = time.time()
    try:
        # pd.read_sql é a ponte simples entre SQLite e DataFrame.
        df = pd.read_sql(generated_sql, conn)

        # Guardamos a lista de colunas para explicar semanticamente o resultado
        # ao modelo de resposta.
        query_columns = [safe_str(str(column)) for column in df.columns.tolist()]

        # O resultado é formatado em modo humano antes de ser enviado ao LLM.
        query_result = format_dataframe_for_prompt(df)
        query_column_context = safe_str(build_query_column_context(generated_sql, query_columns))
        elapsed = time.time() - sql_execution_start_time
        trace_operation(
            "execute_sql_node.resultado",
            sql_chars=len(generated_sql),
            row_count=len(df),
            column_count=len(query_columns),
            query_result_chars=len(query_result),
            query_column_context_chars=len(query_column_context),
            elapsed_s=f"{elapsed:.2f}",
        )
        log_progress(f"(SQLITE)(OK) Executado em {elapsed:.2f}s")
        return {
            "query_result": query_result,
            "query_column_context": query_column_context,
            "error_message": "",
            "sql_execution_time": state.get("sql_execution_time", 0.0) + elapsed,
        }
    except Exception as exc:
        # O próprio erro real do SQLite vira insumo para retry no nó SQL.
        elapsed = time.time() - sql_execution_start_time
        detail = safe_str(str(exc))
        log_progress(f"(SQLITE)(ERRO DE SINTAXE) Encaminhando string de erro para autocorreção da IA: {detail}")
        return {
            "error_message": f"Erro de execução no SQLite: {detail}",
            "query_result": "",
            "query_column_context": "",
            "sql_execution_time": state.get("sql_execution_time", 0.0) + elapsed,
        }

# -----------------------------------------------------------------------------
# 6. NÓ DE RESPOSTA FINAL
# -----------------------------------------------------------------------------
# Este nó pega:
# - a pergunta original
# - o resultado tabular
# - a semântica das colunas
# e transforma isso em uma resposta curta para o operador.
# -----------------------------------------------------------------------------
def respond_node(state: AgentState) -> Dict[str, Any]:
    error_message = safe_str(state.get("error_message", ""))

    # Se chegamos aqui com erro, devolvemos mensagem de falha operacional.
    if error_message:
        return {"final_response": f"Operador, falha crônica no processamento de dados: {error_message}"}

    question = safe_str(state.get("question", ""))
    query_result = safe_str(state.get("query_result", ""))
    query_column_context = safe_str(state.get("query_column_context", ""))

    if not query_result or "Empty DataFrame" in query_result:
        return {"final_response": "Pesquisa concluída, porém a base analítica não retornou registros correspondentes."}

    # Agora o papel do modelo muda:
    # ele deixa de ser "gerador de SQL" e vira "redator técnico-operacional".
    prompt_relatorio = f"""
Atue como um Engenheiro de Produção de Petróleo Sênior da Equinor dando retorno curto para um operador em sala de controle da plataforma Mærsk Inspirer no Campo de Volve.
Interprete a dúvida do operador e os dados estruturados obtidos do banco SQLite.
Entregue uma resposta operacional objetiva, sem tom de relatório executivo.

Pergunta/Dúvida do Operador:
"{question}"

Dicionário Semântico das Colunas Retornadas:
{query_column_context}

Dados Reais do Banco de Dados:
{query_result}

Diretrizes para a Resposta Operacional:
1. Responda inteiramente em português, com clareza e autoridade sênior, mas sem saudações e sem frases como 'Prezado Operador'.
2. A primeira linha deve começar com 'Diagnóstico:' e responder objetivamente ao que foi perguntado.
3. Use linguagem de sala de controle: direta, operacional e verificável. Evite tom de relatório executivo, sumário executivo ou parecer acadêmico.
4. Sempre adicione as unidades físicas corretas (Sm3/d, bar(a), %, etc.) baseando-se nos metadados.
5. Quando houver campos percentuais ou taxas fracionárias no contexto, apresente o valor em formato humano de percentual. Exemplo: 0.15129 = 15.13%.
6. Para perguntas que envolvem previsões, projeções, riscos ou decaimentos futuros, use os indicadores de tendência (_trend_strength), momentum, taxas de variação (_roc_30d) e aceleração para justificar o diagnóstico.
7. Se sugerir ação, cite verificações operacionais concretas como choke, pressão de fundo, intervenção recente, instabilidade de fluxo ou qualidade da medição. Não use recomendação genérica solta.
8. Não invente causas. Diferencie claramente fato observado de hipótese operacional.

Diretrizes de Limitação de Tamanho (Estritas):
1. Vá direto ao ponto. Proibido incluir introduções teóricas, definições conceituais de livros ou explicações sobre o que significa cada indicador (o operador já sabe o que é momentum e aceleração).
2. Use no máximo 2 parágrafos curtos após a linha de diagnóstico, focando apenas no diagnóstico operacional da data de referência.
3. Se incluir tabela, use tabela markdown compacta. Nunca use bloco de código.
4. Limite a resposta final a um teto máximo de 250 a 300 palavras. Seja conciso e cirúrgico.
""".strip()

    log_progress(f"(REMOTE)(RESPOND) Processando parecer de linguagem natural via {REMOTE_TEXT_MODEL}...")
    trace_operation(
        "respond_node.contexto_entrada",
        model=REMOTE_TEXT_MODEL,
        question_chars=len(question),
        query_result_chars=len(query_result),
        query_column_context_chars=len(query_column_context),
        prompt_chars=len(prompt_relatorio),
    )
    response_start_time = time.time()
    try:
        # temperature=0.2 mantém a resposta estável, mas ainda permite redação
        # um pouco mais natural do que a etapa puramente estrutural do SQL.
        response = openrouter_client.chat.completions.create(
            model=REMOTE_TEXT_MODEL,
            messages=[{"role": "user", "content": prompt_relatorio}],
            temperature=0.2,  # Baixa temperatura para manter o rigor técnico e evitar alucinações comerciais
            extra_headers={
                "HTTP-Referer": "petroleo.local",
                "X-Title": "Parecer de Engenharia Volve",
            },
        )
        final_text = extract_openrouter_text(response).strip()
        elapsed = time.time() - response_start_time
        trace_operation(
            "respond_node.contexto_saida",
            model=REMOTE_TEXT_MODEL,
            output_chars=len(final_text),
            elapsed_s=f"{elapsed:.2f}",
        )
        log_progress(f"(REMOTE)(OK) Resposta operacional gerada com sucesso.")
        return {
            "final_response": final_text,
            "remote_response_time": state.get("remote_response_time", 0.0) + elapsed,
            "remote_prompt_chars": state.get("remote_prompt_chars", 0) + len(prompt_relatorio),
        }
    except Exception as exc:
        detail = safe_str(str(exc))
        return {
            "final_response": f"(FALHA REMOTA EM LINGUAGEM NATURAL): O SQL executou, mas a API falhou em gerar o parecer técnico: {detail}\n\nDados brutos recuperados:\n{query_result}"
        }

# -----------------------------------------------------------------------------
# 7. ORQUESTRAÇÃO DO PIPELINE LANGGRAPH
# -----------------------------------------------------------------------------
def should_retry_or_respond(state: AgentState) -> str:
    # Esta função é um roteador.
    # Se ainda há erro e ainda há tentativas disponíveis, volta ao nó SQL.
    # Caso contrário, segue para a resposta final.
    has_error = bool(state.get("error_message"))
    retry_count = state.get("retry_count", 0)

    if has_error and retry_count < MAX_SQL_RETRIES:
        return "generate_sql"

    return "respond"

def build_workflow_app():
    # O LangGraph é montado explicitamente:
    # generate_sql -> execute_sql -> (retry ou respond)
    workflow = StateGraph(AgentState)
    workflow.add_node("generate_sql", generate_sql_node)
    workflow.add_node("execute_sql", execute_sql_node)
    workflow.add_node("respond", respond_node)
    workflow.add_edge(START, "generate_sql")
    workflow.add_edge("generate_sql", "execute_sql")
    workflow.add_conditional_edges(
        "execute_sql",
        should_retry_or_respond,
        {"generate_sql": "generate_sql", "respond": "respond"},
    )
    workflow.add_edge("respond", END)
    return workflow.compile()

def build_initial_state(question: str) -> AgentState:
    # Estado inicial limpo para cada rodada do pipeline.
    return {
        "question": safe_str(question),
        "generated_sql": "",
        "error_message": "",
        "retry_count": 0,
        "query_result": "",
        "query_column_context": "",
        "sql_generation_time": 0.0,
        "sql_execution_time": 0.0,
        "remote_response_time": 0.0,
        "local_prompt_chars": 0,
        "remote_prompt_chars": 0,
        "final_response": "",
    }

def validate_remote_configuration() -> None:
    # Valida se a chave da API foi realmente configurada.
    # Isso evita rodar metade do notebook antes de descobrir que a chave falta.
    cleaned_key = safe_str(OPENROUTER_API_KEY).strip()
    if not cleaned_key or cleaned_key == "SUA_CHAVE_OPENROUTER_AQUI":
        raise ValueError(
            "OPENROUTER_API_KEY nao configurada. Defina a chave antes de executar o pipeline remoto."
        )

def run_pipeline(question: str) -> AgentState:
    # Função de alto nível que executa o grafo inteiro para uma pergunta.
    initial_state = build_initial_state(question)
    pipeline_start_time = time.time()

    # recursion_limit evita loop infinito caso algo saia do previsto.
    output = app.invoke(initial_state, {"recursion_limit": 15})
    elapsed = time.time() - pipeline_start_time
    trace_operation(
        "run_pipeline.resumo",
        total_elapsed_s=f"{elapsed:.2f}",
        sql_generation_time_s=f"{output.get('sql_generation_time', 0.0):.2f}",
        sql_execution_time_s=f"{output.get('sql_execution_time', 0.0):.2f}",
        remote_response_time_s=f"{output.get('remote_response_time', 0.0):.2f}",
        local_prompt_chars=output.get('local_prompt_chars', 0),
        remote_prompt_chars=output.get('remote_prompt_chars', 0),
        generated_sql_chars=len(safe_str(output.get('generated_sql', ''))),
        final_response_chars=len(safe_str(output.get('final_response', ''))),
    )
    log_progress(f"[PIPELINE][FIM] pergunta={question} tempo_total={elapsed:.2f}s")
    return output

# Este dicionário alimenta os logs didáticos automáticos.
# Cada função importante ganha uma explicação curta sobre seu papel.
METHOD_PURPOSES = {
    "resolve_database_path": "Resolve o caminho absoluto do banco SQLite ativo.",
    "build_read_only_sqlite_uri": "Monta a URI de leitura somente para o SQLite.",
    "clean_generated_sql": "Limpa a saída bruta do modelo SQL antes da execução.",
    "format_value_for_prompt": "Formata valores individuais para contexto legível do LLM.",
    "format_dataframe_for_prompt": "Formata o DataFrame inteiro para contexto humano no prompt.",
    "extract_text_content": "Extrai texto útil de respostas estruturadas do provedor remoto.",
    "extract_openrouter_text": "Extrai e valida o texto retornado pela API OpenRouter.",
    "build_sql_generation_error": "Monta o estado de erro da etapa de geração SQL.",
    "build_empty_execution_response": "Monta o estado vazio quando a execução SQL não pode prosseguir.",
    "try_build_rule_based_sql": "Tenta responder perguntas simples com SQL heurístico local.",
    "build_local_sql_context": "Seleciona colunas relevantes para reduzir o contexto do prompt SQL.",
    "build_query_column_context": "Resume semanticamente as colunas retornadas pela consulta.",
    "choose_pipeline_question": "Sorteia a pergunta operacional que será executada no pipeline.",
    "detect_implausible_forecast_sql": "Bloqueia projeções SQL implausíveis antes da execução.",
    "validar_sql_seguro_e_compativel": "Valida segurança e compatibilidade SQLite da query gerada.",
    "build_sql_prompt": "Monta o prompt completo de Text-to-SQL com schema e few-shot.",
    "generate_sql_node": "Gera a query SQL remota ou heurística para responder a pergunta.",
    "execute_sql_node": "Executa a query SQL no banco e prepara o resultado para o LLM.",
    "respond_node": "Gera a resposta operacional final em linguagem natural.",
    "should_retry_or_respond": "Decide entre nova tentativa de SQL ou resposta final.",
    "build_workflow_app": "Compila o grafo LangGraph do pipeline operacional.",
    "build_initial_state": "Cria o estado inicial do pipeline para a pergunta atual.",
    "validate_remote_configuration": "Verifica se a chave da API remota está configurada.",
    "run_pipeline": "Executa o pipeline completo e consolida métricas finais.",
}

# Aqui aplicamos a instrumentação a todas as funções listadas acima.
for method_name, purpose in METHOD_PURPOSES.items():
    instrument_named_method(method_name, purpose)

# Compilamos o app uma única vez ao carregar a célula.
app = build_workflow_app()
log_progress("[PIPELINE] Grafo LangGraph compilado com sucesso.")

PIPELINE_OUTPUT = None
PIPELINE_SELECTED_QUESTION = None
if PIPELINE_AUTO_RUN:
    # Modo de execução automática:
    # 1. valida chave
    # 2. sorteia pergunta
    # 3. executa pipeline
    # 4. mostra SQL e resposta final
    validate_remote_configuration()
    PIPELINE_SELECTED_QUESTION = choose_pipeline_question()
    selected_question = safe_str(PIPELINE_SELECTED_QUESTION["pergunta"])
    selected_category = safe_str(PIPELINE_SELECTED_QUESTION["categoria"])
    selected_index = PIPELINE_SELECTED_QUESTION["indice"]
    log_progress(
        f"[PIPELINE][SORTEIO] total_perguntas={len(PIPELINE_QUESTION_BANK)} categoria={selected_category} indice={selected_index}"
    )

    # Linha visual para separar os logs do início da rodada.
    print("-" * 100, flush=True)
    log_progress(f"[PIPELINE][INICIO] Executando pergunta sorteada: {selected_question}")
    PIPELINE_OUTPUT = run_pipeline(selected_question)
    log_progress(f"[PIPELINE][SQL] {safe_str(PIPELINE_OUTPUT.get('generated_sql', ''))}")
    log_progress("[PIPELINE][RESPOSTA_FINAL]")

    # Linha visual para separar rastros técnicos da mensagem final ao usuário.
    print("-" * 100, flush=True)
    print(safe_str(PIPELINE_OUTPUT.get("final_response", "")), flush=True)
else:
    log_progress("[PIPELINE] Auto-run desabilitado. Use choose_pipeline_question() para sortear ou run_pipeline(<pergunta>) para executar manualmente.")


[DADOS] CSV operacional carregado de: /home/wolf/Documentos/lab-artificial-inteligence/notebooks/10-exercicio-production-surveillance/volve_with_feature_engineering_temporal.csv
[DADOS] Banco SQLite materializado em: /home/wolf/Documentos/lab-artificial-inteligence/notebooks/10-exercicio-production-surveillance/volve_with_feature_engineering_temporal.db
[DADOS] Tabela operacional: volve_with_feature_engineering_temporal
[DADOS] Registros carregados em source_df: 15634
[DEBUG][build_workflow_app] Compila o grafo LangGraph do pipeline operacional.
[TRACE][build_workflow_app.fim] elapsed_s=0.0091
[PIPELINE] Grafo LangGraph compilado com sucesso.
[DEBUG][validate_remote_configuration] Verifica se a chave da API remota está configurada.
[TRACE][validate_remote_configuration.fim] elapsed_s=0.0000
[DEBUG][choose_pipeline_question] Sorteia a pergunta operacional que será executada no pipeline.
[TRACE][choose_pipeline_question.fim] elapsed_s=0.0001
[PIPELINE][SORTEIO] total_perguntas=12 categor